**SQLite Database**

In [1]:
import sqlite3

In [2]:
# create a connection to SQLite DB
conn = sqlite3.connect("pijush-sqllite-sample.db")

In [3]:
# create a cursor to help us execute the SQL commands
cursor = conn.cursor()

In [5]:
# create a table
# stocks : stock_code, stock_name

cursor.execute(
"""
CREATE TABLE IF NOT EXISTS stocks (
  stock_code INTEGER PRIMARY KEY,
  stock_name TEXT NOT NULL
)
"""
)


In [6]:
# insert some data
cursor.execute(
    "INSERT INTO stocks (stock_name) VALUES (?)", ('TESLA',)
)



In [7]:
cursor.execute(
    "INSERT INTO stocks (stock_name) VALUES (?)", ('Microsoft',)
)


In [8]:
# select records

cursor.execute("SELECT * FROM stocks")

In [9]:
rows = cursor.fetchall()
rows

[(1, 'TESLA'), (2, 'Microsoft')]

In [10]:
conn.commit()

In [11]:
conn.close()

**Using SQLite as vector storage**

# vector = array of numbers -> numpy arrays -> [1.2, 2,5, 3,7]
# store the information into a bytes format

In [13]:
import numpy as np

In [14]:
conn = sqlite3.connect("PJsample_vectors.db")

In [15]:
cursor = conn.cursor()

In [16]:
cursor.execute(
"""
CREATE TABLE IF NOT EXISTS vectors (
  vector_id INTEGER PRIMARY KEY,
  vector BLOB NOT NULL
)
"""
)

In [17]:
vect_tsla = np.array([1.3, 3.5, 2.2, 0.9])
vect_msft = np.array([2.8, 1.6, 3.8, 2.2])

In [18]:
type(vect_tsla)

numpy.ndarray

In [19]:
vect_tsla.tobytes()

b'\xcd\xcc\xcc\xcc\xcc\xcc\xf4?\x00\x00\x00\x00\x00\x00\x0c@\x9a\x99\x99\x99\x99\x99\x01@\xcd\xcc\xcc\xcc\xcc\xcc\xec?'

In [21]:
print(sqlite3.Binary(vect_tsla.tobytes()))

In [22]:
cursor.execute("INSERT INTO vectors (vector) VALUES (?)", (sqlite3.Binary(vect_tsla.tobytes()),))

In [23]:
cursor.execute("INSERT INTO vectors (vector) VALUES (?)", (sqlite3.Binary(vect_msft.tobytes()),))

In [24]:
cursor.execute("SELECT * FROM vectors")

In [25]:
rows = cursor.fetchall()
rows

[(1,
  b'\xcd\xcc\xcc\xcc\xcc\xcc\xf4?\x00\x00\x00\x00\x00\x00\x0c@\x9a\x99\x99\x99\x99\x99\x01@\xcd\xcc\xcc\xcc\xcc\xcc\xec?'),
 (2,
  b'ffffff\x06@\x9a\x99\x99\x99\x99\x99\xf9?ffffff\x0e@\x9a\x99\x99\x99\x99\x99\x01@')]

In [ ]:
# Deserialization

In [26]:
rows

[(1,
  b'\xcd\xcc\xcc\xcc\xcc\xcc\xf4?\x00\x00\x00\x00\x00\x00\x0c@\x9a\x99\x99\x99\x99\x99\x01@\xcd\xcc\xcc\xcc\xcc\xcc\xec?'),
 (2,
  b'ffffff\x06@\x9a\x99\x99\x99\x99\x99\xf9?ffffff\x0e@\x9a\x99\x99\x99\x99\x99\x01@')]

In [ ]:
# np.frombuffer

In [27]:
v = np.frombuffer(rows[0][1], dtype=np.float64)
v

array([1.3, 3.5, 2.2, 0.9])

In [28]:
vectors = []
for row in rows:
  v = np.frombuffer(row[1], dtype=np.float64)
  vectors.append(v)

In [29]:
vectors

[array([1.3, 3.5, 2.2, 0.9]), array([2.8, 1.6, 3.8, 2.2])]

In [ ]:
# Find nearest vector

In [30]:
q_vector = np.array([3.5, 2.2, 3.5, 5.5])

In [31]:
cursor.execute("""
SELECT vector FROM vectors ORDER BY abs(vector - ?) DESC
""", (sqlite3.Binary(q_vector.tobytes()),))

In [32]:
res = cursor.fetchone()

In [33]:
res

(b'\xcd\xcc\xcc\xcc\xcc\xcc\xf4?\x00\x00\x00\x00\x00\x00\x0c@\x9a\x99\x99\x99\x99\x99\x01@\xcd\xcc\xcc\xcc\xcc\xcc\xec?',)

In [34]:
np.frombuffer(res[0], dtype=np.float64)

array([1.3, 3.5, 2.2, 0.9])